# DMP Bridge — Run Pipeline

Extract and label a DMP PDF in three steps:

1. **Configure** — choose your PDF, model, and extractor below
2. **Run** — one cell runs the full pipeline
3. **Inspect** — browse the labeled blocks and structured JSON output

In [ ]:
import os
from pathlib import Path

# Navigate to the project root regardless of where Jupyter started.
# Handles: VS Code (cwd = notebooks/), nbconvert (cwd = project root), JupyterLab.
_cwd = Path.cwd()
if _cwd.name == "notebooks" and (_cwd.parent / "dmpbridge").exists():
    os.chdir(_cwd.parent)
elif not (_cwd / "dmpbridge").exists():
    raise RuntimeError(f"Cannot find project root from {_cwd}. Open this notebook from the project directory.")

print(f"Working directory: {Path.cwd()}")

## 1 — Configuration

Edit the values in the cell below, then run all cells.

In [ ]:
# ── Input ─────────────────────────────────────────────────────────────────────
PDF_PATH  = Path("data/input/pdfs/sample3.pdf")   # path to your PDF

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL     = "gemma4:e4b"              # options: "llama3.1:8b"  "llama3.3:70b"  "gemma4:e4b"
HOST      = "http://localhost:11434"  # Ollama server URL

# ── Extractor ─────────────────────────────────────────────────────────────────
EXTRACTOR = "pdfplumber"   # options: "pdfplumber"  "docling"

# ── Annotation rules ──────────────────────────────────────────────────────────
APPLY_RULES = True         # backfill empty question texts from section titles

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR = Path("data/output/pipeline_run")
OUT_DIR.mkdir(parents=True, exist_ok=True)

stem            = PDF_PATH.stem
LABELED_JSON    = OUT_DIR / f"{stem}_labeled.json"
STRUCTURED_JSON = OUT_DIR / f"{stem}_structured.json"

print(f"PDF       : {PDF_PATH}  {'✓ exists' if PDF_PATH.exists() else '✗ NOT FOUND'}")
print(f"Model     : {MODEL}")
print(f"Extractor : {EXTRACTOR}")
print(f"Rules     : {APPLY_RULES}")
print(f"Output    : {OUT_DIR}/")

In [ ]:
import requests

try:
    r = requests.get(f"{HOST}/api/tags", timeout=5)
    r.raise_for_status()
    models_available = [m["name"] for m in r.json().get("models", [])]
    model_ok = any(MODEL in m for m in models_available)
    print(f"Ollama  : running at {HOST}  ✓")
    print(f"Model   : {MODEL}  {'✓ loaded' if model_ok else '✗ not found — run: ollama pull ' + MODEL}")
    if not model_ok:
        print(f"Available models: {models_available}")
except Exception as e:
    print(f"Ollama  : NOT reachable at {HOST}  ✗")
    print(f"  → Start Ollama first, then re-run this cell.")
    print(f"  → Install: https://ollama.com")
    raise SystemExit("Ollama must be running before you can run the pipeline.")

## 2 — Run the pipeline

This calls the full pipeline: **extract → classify → convert → (optionally) apply annotation rules.**

In [ ]:
import dmpbridge

blocks = dmpbridge.process_pdf(
    PDF_PATH,
    model=MODEL,
    host=HOST,
    extractor=EXTRACTOR,
    apply_rules=APPLY_RULES,
    output=LABELED_JSON,
    structured_output=STRUCTURED_JSON,
    raw_dir=None,
)

print(f"Done — {len(blocks)} blocks labeled")
print(f"Labeled JSON    → {LABELED_JSON}")
print(f"Structured JSON → {STRUCTURED_JSON}")

## 3 — Inspect labeled blocks

Each block shows the text from the PDF and the label the model assigned.

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "page":       b["page"],
        "label":      b.get("label", "—"),
        "confidence": round(b.get("confidence", 1.0), 2),
        "bold":       b.get("is_bold", False),
        "text":       b["text"][:120],
    }
    for b in blocks
])

LABEL_COLORS = {
    "title":               "background-color: #fef9c3",
    "section.title":       "background-color: #dbeafe",
    "section.description": "background-color: #ede9fe",
    "question.text":       "background-color: #dcfce7",
    "answer.text":         "background-color: #f1f5f9",
}

def color_row(row):
    return [LABEL_COLORS.get(row["label"], "")] * len(row)

df.style.apply(color_row, axis=1)

In [ ]:
print("Label distribution:")
print(df["label"].value_counts().to_string())

## 4 — Inspect structured JSON output

The structured JSON nests blocks into the DMP Tool schema:
`narrative → template → section[] → question[] → answer`

In [ ]:
import json

structured = json.loads(STRUCTURED_JSON.read_text(encoding="utf-8"))
template   = structured["narrative"]["template"]

print(f"Title    : {template.get('title', '—')}")
print(f"Sections : {len(template.get('section', []))}")
print()

for sec in template.get("section", []):
    print(f"  [{sec['order']}] {sec['title']}")
    if sec.get("description"):
        print(f"      desc: {sec['description'][:80]}")
    for q in sec.get("question", []):
        ans = q.get("answer", {}).get("json", {}).get("answer", "")
        print(f"      Q{q['order']}: {q['text'][:80]}")
        print(f"         A: {ans[:100]}")

In [ ]:
# Full structured JSON (scroll to browse)
print(json.dumps(structured, indent=2, ensure_ascii=False))